# E27 — o preço do conserto

O caderno E26 mediu a abertura: quem nunca esquece tem memória igual à idade, e aos catorze mil
dias pede mais dias para aprender do que a série tem. Falta o outro lado — **o que custa escolher
uma memória em vez de deixar que ela envelheça**, que é o que fecha o arco do capítulo.

Quatro estimadores, a mesma série com um degrau, o mesmo horizonte: a média acumulada (a memória
que ninguém escolheu), a exponencial da memória vencedora do capítulo 14, a acumulada que se
reinicia a cada janela, e a janela deslizante.

In [1]:
# <- brinque com: QUANDO, FATOR, HORIZONTE, JANELA, SEMENTES
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import frevolab
from frevolab import esquecimento, graficos, mudanca

RAIZ = Path.cwd()
QUANDO, FATOR = 6000, 2.0
HORIZONTE, JANELA = 250, 250
SEMENTES = 40
DIAS = 14000

print("frevolab %s | degrau no dia %d | horizonte %d | janela %d | %d mundos"
      % (frevolab.VERSAO, QUANDO, HORIZONTE, JANELA, SEMENTES))

frevolab 0.1.0 | degrau no dia 6000 | horizonte 250 | janela 250 | 40 mundos


In [2]:
# Os quatro estimadores, medidos no mesmo horizonte depois do degrau.
linhas = []
for semente in range(SEMENTES):
    x = np.abs(mudanca.degrau(DIAS, np.random.default_rng(500 + semente), fator=FATOR, quando=QUANDO))
    # A verdade tem a ESCALA DA SERIE, e nao uma escala inventada: o nivel e o que a serie mede
    # antes do degrau. Comparar com 1,0 dava noventa e nove por cento de erro para os quatro
    # estimadores --- e a igualdade dos quatro era o sinal de que a conta estava errada.
    base = float(np.median(x[:QUANDO]))
    verdade = np.array([base * (FATOR if t >= QUANDO else 1.0) for t in range(DIAS)])
    series = {
        "a média acumulada": esquecimento.media_acumulada(x),
        "a exponencial da memória de vinte e um": esquecimento.exponencial(x, 1.0 / 21.0),
        "a acumulada que reinicia": esquecimento.janela(esquecimento.media_acumulada(x), JANELA),
        "a janela de duzentos e cinquenta": esquecimento.janela(x, JANELA),
    }
    linhas.append({nome: esquecimento.erro_varios([s], verdade, QUANDO, HORIZONTE)[0]
                   for nome, s in series.items()})
tabela = pd.DataFrame(linhas)
print(tabela.mean().round(4).to_string())
print()
print("pior dos quatro: %s | melhor: %s"
      % (tabela.mean().idxmax(), tabela.mean().idxmin()))
idade = QUANDO
print("a media acumulada chega ao degrau com %d dias de idade: %.0f dias para aprender o nível novo"
      % (idade, esquecimento.dias_da_idade(idade)))

a média acumulada                         0.3948
a exponencial da memória de vinte e um    0.1927
a acumulada que reinicia                  0.4030
a janela de duzentos e cinquenta          0.1742

pior dos quatro: a acumulada que reinicia | melhor: a janela de duzentos e cinquenta
a media acumulada chega ao degrau com 6000 dias de idade: 7223 dias para aprender o nível novo


In [3]:
# Figura 1: os quatro estimadores depois do degrau.
x = np.abs(mudanca.degrau(DIAS, np.random.default_rng(500), fator=FATOR, quando=QUANDO))
BASE = float(np.median(x[:QUANDO]))
fig, eixo = plt.subplots(figsize=(8.8, 4.3))
for nome, serie, cor in (("média acumulada", esquecimento.media_acumulada(x), "#b03a2e"),
                         ("exponencial de 21 dias", esquecimento.exponencial(x, 1.0 / 21.0), "#1f4e79"),
                         ("acumulada que reinicia em 250", esquecimento.janela(esquecimento.media_acumulada(x), JANELA), "#2e7d32"),
                         ("janela de 250 dias", esquecimento.janela(x, JANELA), "#555555")):
    eixo.plot(range(QUANDO - 50, QUANDO + HORIZONTE), np.asarray(serie)[QUANDO - 50:QUANDO + HORIZONTE],
              lw=1.7, color=cor, label=nome)
# A verdade tem a ESCALA DA SERIE: desenha-la em FATOR punha a linha em 2,00 enquanto as curvas
# vivem na casa de 0,0003, e o desenho dizia que todos os estimadores erram tudo.
eixo.axhline(BASE * FATOR, color="#333333", ls=":", lw=1.2, label="a verdade depois do degrau")
eixo.axvline(QUANDO, color="#333333", ls="--", lw=1.1)
eixo.set_xlabel("dias")
eixo.set_ylabel("estimativa do nível")
eixo.legend(frameon=False, fontsize=8)
eixo.grid(alpha=0.25)
fig.tight_layout()
graficos.salvar(fig, "E27_conserto", 1)
plt.close(fig)
print("figura gravada")

figura gravada


## Leitura visual das figuras

Feita nesta sessão abrindo o .png com a ponte de visão (AGENTS.md §9), depois de o caderno rodar.

O que o desenho mostra: o eixo horizontal cobre trezentos dias em torno do degrau e o vertical vai
de 0,006 a 0,020; a linha pontilhada é a verdade depois do degrau, em torno de 0,0136, e a tracejada
vertical marca o dia do degrau. Depois dele as quatro curvas se separam: a azul (a exponencial de
vinte e um dias) reage depressa e oscila em volta da verdade; a cinza (a janela de duzentos e
cinquenta) sobe em rampa, porque a janela vai sendo preenchida por dias novos; e a vermelha (a média
acumulada) e a verde (a acumulada que reinicia) ficam praticamente paradas, na casa de 0,008 ---
carregam tanto passado que o degrau quase não as move dentro do horizonte.

A expectativa que eu havia escrito aqui dizia que a curva verde subiria em degraus. O desenho mostra
o contrário: dentro desta janela ela fica parada, e é justamente isso que o capítulo quer mostrar ---
a memória que ninguém escolheu não aprende no horizonte em que a decisão acontece.

In [4]:
# O resultado: um objeto por grandeza, para o livro citar por comando.
NOMES = {"a média acumulada": "acumulada", "a exponencial da memória de vinte e um": "exponencial",
         "a acumulada que reinicia": "reinicia", "a janela de duzentos e cinquenta": "janela"}
resultado = {
    "conserto_dias": int(DIAS), "conserto_quando": int(QUANDO), "conserto_horizonte": int(HORIZONTE),
    "conserto_janela": int(JANELA), "conserto_mundos": int(SEMENTES),
    "conserto_fator": float(FATOR),
    "conserto_idade_no_degrau": int(QUANDO),
    "conserto_dias_da_idade": float(esquecimento.dias_da_idade(QUANDO)),
}
for nome, curto in NOMES.items():
    resultado["conserto_erro_%s" % curto] = float(tabela[nome].mean())
resultado["conserto_razao_pior_melhor"] = float(tabela.mean().max() / tabela.mean().min())
caminho = Path("lab/resultados/E27_conserto.json")
caminho.write_text(json.dumps(resultado, indent=1, ensure_ascii=False, sort_keys=True), encoding="utf-8")
print("%s gravado | %d grandezas" % (caminho, len(resultado)))

lab/resultados/E27_conserto.json gravado | 13 grandezas
